In [1]:
import wandb

import os
import ast
import wandb
import argparse
import numpy as np
from typing import List
import matplotlib.pyplot as plt
from keras.utils import to_categorical
from utils.data_loader import load_dataset
from ann.neural_network import NeuralNetwork
from sklearn.model_selection import train_test_split;

In [2]:
import random

import wandb
run = wandb.init(
    entity="id25s027-iit-madras",
    project="da6401-assignment1-puneet-id25s027",
    config={
        "learning_rate": 0.02,
        "architecture": "CNN",
        "dataset": "CIFAR-100",
        "epochs": 10,
    },
)

epochs = 10
offset = random.random() / 5
for epoch in range(2, epochs):
    acc = 1 - 2**-epoch - random.random() / epoch - offset
    loss = 2**-epoch + random.random() / epoch + offset
    run.log({"acc": acc, "loss": loss})
run.finish()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\brpun\_netrc
wandb: Currently logged in as: id25s027 (id25s027-iit-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


acc,▁▄▇▇▇██▇
loss,█▆▃▂▂▃▂▁
acc,0.89198
loss,0.01061


In [3]:
PROJECT_NAME = "da6401-assignment1-puneet-id25s027"

In [4]:
dataset = "mnist"

(train_images, train_labels), (test_images, test_labels) = load_dataset(dataset)

train_images, test_images = train_images / 255.0, test_images / 255.0
train_images = train_images.reshape((-1, 28 * 28))
test_images = test_images.reshape((-1, 28 * 28))

output_size = len(np.unique(train_labels).tolist())

print(train_images[0].shape[0])
train_labels = to_categorical(train_labels)
test_labels = to_categorical(test_labels)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 9s 1us/step
784


In [5]:
def train_with_wandb_for_sweep(config=None):
    x_train, x_val, y_train, y_val = train_test_split(train_images, train_labels, test_size = 0.1, random_state = 42)

    with wandb.init(project=PROJECT_NAME, config=config):
        cfg = wandb.config

        output_activation = "softmax"
        if cfg.loss == "mse":
            output_activation = "sigmoid"

        cfg.wandb_project = PROJECT_NAME
        
        model = NeuralNetwork(
            cli_args=cfg,
            input_size = train_images[0].shape[0],
            output_size = output_size,
            output_activation = output_activation
        )

        epochs = cfg.epochs

        model.train(x_train, y_train, epochs=epochs, batch_size=cfg.batch_size, x_val=x_val, y_val=y_val, wandb=wandb)

        test_eval = model.evaluate(test_images, test_labels)
        
        test_log = {
            "test_accuracy": test_eval["accuracy"], 
            "test_recall": test_eval["recall"],
            "test_precision": test_eval["precision"],
            "test_f1": test_eval["f1"],
        }
        wandb.log(test_log)


In [6]:
sweep_config = {
    "method": "random",
    "metric": {'name': 'final_val_loss', 'goal': 'minimize'},
    "parameters": {
        "learning_rate": {
            "values": [0.1, 0.01, 0.0001, 0.001]
        },
        "optimizer": {
            "values": ["sgd", 'adam', 'nadam', "momentum", "nag", "rmsprop"],
        },
        
        "activation": {
            "values": ["relu", "sigmoid", "tanh"]
        },
        "loss": {
            "values": ["cross_entropy", "mse"]
        },
        "weight_init": {
            "values": ["random", "xavier", "zero"]
        },
        "epochs": {
            "values": [10,20,25,30,40,50]
        },
        "hidden_layers": {
            "values": [
                [16,16,16,16,16],
                [32,32,32,32,32],
                [64,64,64,64,64],
                [128,128,128,128],
                [128,128,128,128,128],
                [32,64,128,64,32]
            ]
        },
        "weight_decay": {
            "values": [0.0, 0.01, 0.001, 0.0001]
        },
        "batch_size": {
            "values":[8,16,32,64]
        },
    }
}

In [7]:
sweep_id = wandb.sweep(
    sweep_config,
    project=PROJECT_NAME
)

Create sweep with ID: vqrvqabq
Sweep URL: https://wandb.ai/id25s027-iit-madras/da6401-assignment1-puneet-id25s027/sweeps/vqrvqabq


In [8]:
total_runs = 100
wandb.agent(sweep_id, train_with_wandb_for_sweep, count=total_runs)

wandb: Agent Starting Run: c4p4v5k8 with config:
wandb: 	activation: sigmoid
wandb: 	batch_size: 16
wandb: 	epochs: 50
wandb: 	hidden_layers: [64, 64, 64, 64, 64]
wandb: 	learning_rate: 0.0001
wandb: 	loss: mse
wandb: 	optimizer: sgd
wandb: 	weight_decay: 0.01
wandb: 	weight_init: random
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\brpun\_netrc.


Epoch   1/50 : Train Loss = 0.0909, Train Accuracy = 11.274074074074074 | Val Loss = 0.0909, Val Accuracy = 10.9000
Epoch   6/50 : Train Loss = 0.0900, Train Accuracy = 11.274074074074074 | Val Loss = 0.0900, Val Accuracy = 10.9000
Epoch  11/50 : Train Loss = 0.0900, Train Accuracy = 11.274074074074074 | Val Loss = 0.0900, Val Accuracy = 10.9000
Epoch  16/50 : Train Loss = 0.0900, Train Accuracy = 11.274074074074074 | Val Loss = 0.0900, Val Accuracy = 10.9000
Epoch  21/50 : Train Loss = 0.0900, Train Accuracy = 11.274074074074074 | Val Loss = 0.0900, Val Accuracy = 10.9000
Epoch  26/50 : Train Loss = 0.0900, Train Accuracy = 11.274074074074074 | Val Loss = 0.0900, Val Accuracy = 10.9000
Epoch  31/50 : Train Loss = 0.0900, Train Accuracy = 11.274074074074074 | Val Loss = 0.0900, Val Accuracy = 10.9000
Epoch  36/50 : Train Loss = 0.0900, Train Accuracy = 11.274074074074074 | Val Loss = 0.0900, Val Accuracy = 10.9000
Epoch  41/50 : Train Loss = 0.0900, Train Accuracy = 11.274074074074074 

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


best_val_loss,▁
convergence_epoch,▁
epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
final_train_accuracy,▁
final_train_loss,▁
final_val_accuracy,▁
final_val_loss,▁
overfitting_gap,█▅▃▂▂▃▂▂▁▂▂▂▃▂▂▃▁▂▂▂▂▂▂▃▃▃▃▂▂▁▂▂▁▂▁▂▁▁▂▁
test_accuracy,▁
test_f1,▁
+7,...


wandb: Agent Starting Run: 8la999ao with config:
wandb: 	activation: tanh
wandb: 	batch_size: 32
wandb: 	epochs: 20
wandb: 	hidden_layers: [64, 64, 64, 64, 64]
wandb: 	learning_rate: 0.01
wandb: 	loss: cross_entropy
wandb: 	optimizer: adam
wandb: 	weight_decay: 0.01
wandb: 	weight_init: xavier
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\brpun\_netrc.


...

Epoch   6/50 : Train Loss = nan, Train Accuracy = 9.812962962962963 | Val Loss = nan, Val Accuracy = 10.4000
Epoch  11/50 : Train Loss = nan, Train Accuracy = 9.812962962962963 | Val Loss = nan, Val Accuracy = 10.4000
Epoch  16/50 : Train Loss = nan, Train Accuracy = 9.812962962962963 | Val Loss = nan, Val Accuracy = 10.4000
Epoch  21/50 : Train Loss = nan, Train Accuracy = 9.812962962962963 | Val Loss = nan, Val Accuracy = 10.4000
Epoch  26/50 : Train Loss = nan, Train Accuracy = 9.812962962962963 | Val Loss = nan, Val Accuracy = 10.4000
Epoch  31/50 : Train Loss = nan, Train Accuracy = 9.812962962962963 | Val Loss = nan, Val Accuracy = 10.4000
Epoch  36/50 : Train Loss = nan, Train Accuracy = 9.812962962962963 | Val Loss = nan, Val Accuracy = 10.4000
Epoch  41/50 : Train Loss = nan, Train Accuracy = 9.812962962962963 | Val Loss = nan, Val Accuracy = 10.4000
Epoch  46/50 : Train Loss = nan, Train Accuracy = 9.812962962962963 | Val Loss = nan, Val Accuracy = 10.4000


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch  50/50 : Train Loss = nan, Train Accuracy = 9.812962962962963 | Val Loss = nan, Val Accuracy = 10.4000


convergence_epoch,▁
epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
final_train_accuracy,▁
final_val_accuracy,▁
overfitting_gap,▁
test_accuracy,▁
test_f1,▁
test_precision,▁
test_recall,▁
train_accuracy,▁███████████████████████████████████████
+7,...


<Figure size 640x480 with 0 Axes>